# 07-1 · The LLM-judge proxy

| | |
|---|---|
| **What it does** | an ensemble of three LLMs from three different families rates each group recommendation for perceived fairness, satisfaction and consensus |
| **Proves** | that the selector ranking produced by the arithmetic survives a second, independent, stated judgement procedure |
| **Writes** | `figdata/table7_judge.csv`, `figdata/table8_judge_validation.csv` |
| **Needs** | `GROQ_API_KEY` (or any OpenAI-compatible endpoint). Without a key the notebook loads the cached per-call log instead. |

### Read this before you read any number below

This is **not a human study.** No person judged any recommendation here. It is
a proxy, and every table it writes must be labelled as one in the paper — in
the caption, in the methods, and in the limitations. The claim we are entitled
to make is narrow and checkable:

> *A stated, versioned, three-family LLM ensemble ranks the selectors in this
> order, with this much disagreement between models and this much instability
> across repeats.*

The claim we are **not** entitled to make is "users perceived" anything.

In [1]:
# Locate the repo root (nbio.py lives there) regardless of where this notebook sits.
import os, sys
_root = os.getcwd()
for _ in range(4):
    if os.path.isfile(os.path.join(_root, "nbio.py")): break
    _root = os.path.dirname(_root)
if _root not in sys.path: sys.path.insert(0, _root)
os.chdir(_root)   # figdata/ paths below are repo-relative

In [2]:
import nbio
nbio.bootstrap()

repo     : /project/painting/project-research/AnacodicAI-Labs/code/production-repos/ai-in-food/research_fair-mediator
data     : data  (ingredient-flavor.csv: found, 5 menu files)


PosixPath('/project/painting/project-research/AnacodicAI-Labs/code/production-repos/ai-in-food/research_fair-mediator')

## Step 1 — parameters, stated once

`N_GROUPS` and `K` are deliberately small so this notebook runs in a few
minutes on one key. The paper's numbers come from the full sweep, which is the
same code with the two knobs raised — see the last cell.

In [3]:
SEED       = 7
N_GROUPS   = 3                     # paper run: 60
K          = 5                     # self-consistency repeats (control 3) -- do not lower
TEMPERATURE = 0.0
SELECTORS  = ["average", "least_misery", "most_pleasure", "fair"]

# Control 1: three DIFFERENT families. Two sizes of one model is not an ensemble.
MODELS = ["openai/gpt-oss-20b", "qwen/qwen3.8-27b", "groq/compound"]

COST_CEILING = 5.00                # USD; the run aborts rather than overspending
ORDER_TOLERANCE = 1.0              # control 2: discard an item moving > 1 point on reversal
UNSTABLE_SD = 1.0                  # control 3: sd above this = unstable item

n_calls = N_GROUPS * len(SELECTORS) * len(MODELS) * K * 2
print(f"{N_GROUPS} groups x {len(SELECTORS)} selectors x {len(MODELS)} models x k={K} x 2 orders")
print(f"= {n_calls} calls")

3 groups x 4 selectors x 3 models x k=5 x 2 orders
= 360 calls


## Step 2 — the prompt, frozen before any rating was collected

Control 5. The prompt lives in `nbio.py` under a version string. If it
changes, the version bumps and **every rating is re-collected from scratch** —
otherwise "we tried prompts until the result looked good" is an unanswerable
objection.

In [4]:
print("prompt version:", nbio.JUDGE_PROMPT_VERSION)
print()
print(nbio.JUDGE_PROMPT)

prompt version: v1

You are evaluating how a group of diners would feel about a restaurant
recommendation made on their behalf.

THE GROUP
<<GROUP>>

THE RECOMMENDATION
<<OPTION>>

HOW WELL IT MATCHES EACH DINER (0.0 = poor match, 1.0 = perfect match)
<<SATS>>

Rate the recommendation on three 1-7 scales, judging how the GROUP would
perceive it, not whether the numbers are arithmetically optimal:
  fairness      1 = deeply unfair to someone, 7 = everyone treated fairly
  satisfaction  1 = the group would be unhappy, 7 = the group would be pleased
  consensus     1 = the group would argue, 7 = the group would readily agree

Reply with ONLY a JSON object, no prose, no markdown fence:
{"fairness": <1-7>, "satisfaction": <1-7>, "consensus": <1-7>, "rationale": "<one sentence>"}


## Step 3 — connect a judge

Any OpenAI-compatible endpoint works. Groq is the default because it is fast
and the team already has a key. If no key is present the notebook falls back
to the committed per-call log so the tables below still rebuild.

In [5]:
from openai import OpenAI

KEY = os.environ.get("GROQ_API_KEY", "")
LIVE = bool(KEY)
client = OpenAI(api_key=KEY, base_url="https://api.groq.com/openai/v1") if LIVE else None
print("live API:" , LIVE, "| models:", ", ".join(MODELS) if LIVE else "(cached log)")

live API: False | models: (cached log)


## Step 4 — one item, end to end

Before running hundreds of calls, look at exactly one: the group, the pick the
arithmetic made, the prompt the judge actually sees, and the reply it actually
returns. If this cell looks wrong, nothing after it is worth reading.

In [6]:
catalog = nbio.load_catalog()
groups  = nbio.generate_groups(N_GROUPS, seed=SEED, size_range=(3, 4))

g   = groups[0]
res = nbio.run_group(g, catalog, method="fair")
opt = next(o for o in res["_F"] if o["option_id"] == res["pick_id"])
sats = res["_S"][res["pick_id"]]

print("GROUP:", ", ".join(m["display_name"] for m in g["members"]),
      "| adversarial:", g["adversarial"])
print("PICK :", opt["name"], f"(min_sat={res['min_sat']:.3f}, jain={res['jain']:.3f})")
print("\n--- prompt the judge sees ---")
print(nbio._judge_render(g, opt, sats))

GROUP: Harper, Abigail, Kyle, Grace | adversarial: True
PICK : Delicia Multi Cuisine Restaurant (min_sat=0.418, jain=0.918)

--- prompt the judge sees ---
You are evaluating how a group of diners would feel about a restaurant
recommendation made on their behalf.

THE GROUP
  - Harper: diet non-veg, allergies shellfish
  - Abigail: diet non-veg, allergies shellfish
  - Kyle: diet non-veg, allergies shellfish
  - Grace: diet veg, allergies gluten

THE RECOMMENDATION
  Delicia Multi Cuisine Restaurant

HOW WELL IT MATCHES EACH DINER (0.0 = poor match, 1.0 = perfect match)
  - Harper: 0.46
  - Abigail: 0.85
  - Kyle: 0.42
  - Grace: 0.74

Rate the recommendation on three 1-7 scales, judging how the GROUP would
perceive it, not whether the numbers are arithmetically optimal:
  fairness      1 = deeply unfair to someone, 7 = everyone treated fairly
  satisfaction  1 = the group would be unhappy, 7 = the group would be pleased
  consensus     1 = the group would argue, 7 = the group would rea

In [7]:
if LIVE:
    one = nbio.judge_recommendation(g, opt, sats, client=client,
                                    model=MODELS[0], temperature=TEMPERATURE)
    print("--- raw reply ---"); print(one["raw"])
    print("\nparsed ok:", one["ok"],
          "| fairness", one["fairness"], "satisfaction", one["satisfaction"],
          "consensus", one["consensus"])
    print("rationale:", one["rationale"])
else:
    print("skipped (no key)")

skipped (no key)


### What happens when the judge misbehaves

A malformed reply, a rating outside 1–7, or a dead API returns `ok=False` and
the item is **dropped and counted**. It is never quietly imputed to the
midpoint — a 4/7 we invented would be indistinguishable in the table from a
4/7 a model actually chose, and the drop rate is the honest thing to report.

In [8]:
class _Broken:                                    # a judge that returns nonsense
    class chat:
        class completions:
            @staticmethod
            def create(**kw):
                return type("R", (), {"choices": [type("C", (), {"message": type(
                    "M", (), {"content": "sure! fairness is about 9 I guess"})()})()]})()

bad = nbio.judge_recommendation(g, opt, sats, client=_Broken, model="broken")
print("ok:", bad["ok"], "| fairness:", bad["fairness"], "| raw:", bad["raw"][:60])
assert bad["ok"] is False and bad["fairness"] is None
print("-> dropped and counted, not imputed")

ok: False | fairness: None | raw: sure! fairness is about 9 I guess
-> dropped and counted, not imputed


## Step 5 — the run

Every call is logged to `runs/` **before** anything is aggregated, so a crash
half way through costs money but not data, and the aggregation can be redone
without re-paying for the calls.

In [9]:
import json, time

LOG = os.path.join("runs", f"judge_{nbio.JUDGE_PROMPT_VERSION}_seed{SEED}.jsonl")
os.makedirs("runs", exist_ok=True)

if LIVE:
    meter = nbio.cost_meter(COST_CEILING)
    t0, rows = time.time(), []
    with open(LOG, "w") as fh:
        for g in groups:
            for sel in SELECTORS:
                res = nbio.run_group(g, catalog, method=sel)
                if res is None:
                    continue                       # infeasible group: nothing to judge
                opt  = next(o for o in res["_F"] if o["option_id"] == res["pick_id"])
                sats = res["_S"][res["pick_id"]]
                batch = nbio.judge_ensemble(g, opt, sats, client=client, models=MODELS,
                                            k=K, temperature=TEMPERATURE, meter=meter)
                for r in batch:
                    r.update(selector=sel, min_sat=res["min_sat"], jain=res["jain"],
                             mean_sat=res["mean_sat"], option_name=opt["name"])
                    fh.write(json.dumps(r) + "\n")
                rows += batch
            print(f"  {g['group_id']} done  ({len(rows)} calls, {time.time()-t0:.0f}s)")
    print(f"\n{len(rows)} calls in {time.time()-t0:.0f}s -> {LOG}")
    print(meter.report())
else:
    rows = [json.loads(l) for l in open(LOG)]
    print(f"loaded {len(rows)} cached calls from {LOG}")

loaded 360 cached calls from runs/judge_v1_seed7.jsonl


## Step 6 — the three controls, checked out loud

None of these are optional. Each one names a way this result could be fake.

In [10]:
import pandas as pd
df = pd.DataFrame(rows)
ok = df[df.ok]

print(f"CONTROL 0 -- drop rate: {1 - len(ok)/len(df):.3%}  ({len(df)-len(ok)} of {len(df)} calls)")
print()
print("CONTROL 1 -- inter-model agreement on the selector ranking (Spearman)")
rank = ok.groupby(["model", "selector"]).fairness.mean().unstack()
print(rank.round(3).to_string())
import itertools
rhos = []
for a, b in itertools.combinations(rank.index, 2):
    rho = nbio.spearman(rank.loc[a, SELECTORS].tolist(), rank.loc[b, SELECTORS].tolist())
    rhos.append(rho)
    print(f"  rho({a}, {b}) = {rho:+.3f}")
print(f"  mean pairwise rho = {sum(rhos)/len(rhos):+.3f}")
if min(rhos) < 0:
    print("  -> AT LEAST ONE PAIR DISAGREES ON THE ORDERING (negative rho).")
    print("     The pooled ranking in Table 7 is therefore an average over models")
    print("     that do not individually agree. This must be stated in the paper;")
    print("     it is the single most attackable part of the proxy.")

CONTROL 0 -- drop rate: 0.000%  (0 of 360 calls)

CONTROL 1 -- inter-model agreement on the selector ranking (Spearman)
selector            average   fair  least_misery  most_pleasure
model                                                          
groq/compound         4.000  4.333         4.233          4.167
openai/gpt-oss-20b    3.600  3.567         3.567          3.467
qwen/qwen3.8-27b      3.833  3.833         3.833          3.500
  rho(groq/compound, openai/gpt-oss-20b) = -0.316
  rho(groq/compound, qwen/qwen3.8-27b) = +0.258
  rho(openai/gpt-oss-20b, qwen/qwen3.8-27b) = +0.816
  mean pairwise rho = +0.253
  -> AT LEAST ONE PAIR DISAGREES ON THE ORDERING (negative rho).
     The pooled ranking in Table 7 is therefore an average over models
     that do not individually agree. This must be stated in the paper;
     it is the single most attackable part of the proxy.


In [11]:
print("CONTROL 2 -- order / position bias")
o = ok.groupby(["group_id", "selector", "reverse_order"])[["fairness","satisfaction","consensus"]].mean()
gaps = (o.xs(False, level="reverse_order") - o.xs(True, level="reverse_order")).abs()
flagged = (gaps > ORDER_TOLERANCE).any(axis=1)
print(f"  mean |forward - reversed|: " +
      ", ".join(f"{d} {gaps[d].mean():.3f}" for d in gaps.columns))
print(f"  items moving > {ORDER_TOLERANCE} point on reversal: {flagged.sum()} of {len(gaps)}  (discarded)")
DISCARDED = set(gaps.index[flagged])

CONTROL 2 -- order / position bias
  mean |forward - reversed|: fairness 0.300, satisfaction 0.172, consensus 0.356
  items moving > 1.0 point on reversal: 0 of 12  (discarded)


In [12]:
print(f"CONTROL 3 -- self-consistency across k={K} repeats")
sd = ok.groupby(["group_id","selector","model"])[["fairness","satisfaction","consensus"]].std()
print("  mean sd per dimension: " + ", ".join(f"{d} {sd[d].mean():.3f}" for d in sd.columns))
unstable = (sd > UNSTABLE_SD).any(axis=1)
print(f"  unstable cells (sd > {UNSTABLE_SD}): {unstable.sum()} of {len(sd)}  ({unstable.mean():.1%})")

CONTROL 3 -- self-consistency across k=5 repeats
  mean sd per dimension: fairness 0.303, satisfaction 0.319, consensus 0.388
  unstable cells (sd > 1.0): 0 of 36  (0.0%)


## Step 7 — Table 7: the selector ranking, as the proxy sees it

The column that matters is not any single mean — it is whether the proxy's
ordering matches the arithmetic ordering from Table 1. Agreement strengthens
the main claim; disagreement is a finding in its own right and gets reported,
not smoothed.

In [13]:
keep = ok[~ok.set_index(["group_id","selector"]).index.isin(DISCARDED)]

# The unit of analysis is the ITEM (one group x one selector), not the call.
# Averaging over all 90 calls would treat k repeats and 3 models of the same
# item as independent observations and shrink the CI by roughly sqrt(30) --
# pseudo-replication. Collapse to one mean per item first, then take the CI
# across items.
out = []
for sel in SELECTORS:
    s = keep[keep.selector == sel]
    per_item = s.groupby("group_id")[["fairness","satisfaction","consensus"]].mean()
    row = {"selector": sel, "n_items": len(per_item), "n_calls": len(s)}
    for d in ("fairness", "satisfaction", "consensus"):
        m, ci, _ = nbio.mean_ci(per_item[d].tolist())
        row[f"{d}_mean"], row[f"{d}_ci95"] = round(m, 3), round(ci, 3)
    row["drop_rate"] = round(1 - len(s) / max(1, len(df[df.selector == sel])), 4)
    out.append(row)

t7 = pd.DataFrame(out).sort_values("fairness_mean", ascending=False)
t7.to_csv("figdata/table7_judge.csv", index=False)
print("judge ranking (most fair first):", " > ".join(t7.selector))

# Is the ordering actually separated, or is it noise?
top, second = t7.iloc[0], t7.iloc[1]
gap = top.fairness_mean - second.fairness_mean
overlap = gap < (top.fairness_ci95 + second.fairness_ci95)
print(f"top-two gap = {gap:.3f} on a 7-point scale; CIs overlap: {overlap}")
if overlap:
    print("  -> the ORDER is not statistically separated at this N. Report the")
    print("     ordering as indicative and raise N_GROUPS before claiming it.")
t7

judge ranking (most fair first): fair > least_misery > average > most_pleasure
top-two gap = 0.033 on a 7-point scale; CIs overlap: True
  -> the ORDER is not statistically separated at this N. Report the
     ordering as indicative and raise N_GROUPS before claiming it.


,selector,n_items,n_calls,fairness_mean,fairness_ci95,satisfaction_mean,satisfaction_ci95,consensus_mean,consensus_ci95,drop_rate
3,fair,3,90,3.911,0.421,4.244,0.157,3.400,0.425,0.0
1,least_misery,3,90,3.878,0.321,4.233,0.285,3.378,0.350,0.0
0,average,3,90,3.811,0.257,4.222,0.230,3.300,0.196,0.0
2,most_pleasure,3,90,3.711,0.492,4.200,0.302,3.233,0.309,0.0


## Step 8 — Table 8: does the judge agree with published human findings?

Control 4, and the one that separates a judge that is merely self-consistent
from one that is pointing in the right direction. We take directional claims
from the published human studies — *"humans rate X fairer than Y"* — and check
only the **direction**. Magnitudes are not expected to match.

`data/judge/human_claims.csv` carries a `verified` column. Every claim ships as
`pending` until somebody opens the paper and fills in `page_cite`. **A claim
still marked `pending` must not appear in the submission.**

In [14]:
claims = pd.read_csv("data/judge/human_claims.csv")
means  = t7.set_index("selector").fairness_mean

rows8 = []
for _, c in claims.iterrows():
    a, b = c.test_selector_a, c.test_selector_b
    if a not in means or b not in means:
        continue
    judge_dir = "a_fairer" if means[a] > means[b] else "b_fairer"
    rows8.append({"claim_id": c.claim_id, "source_paper": c.source_paper,
                  "comparison": f"{a} vs {b}",
                  "human_direction": c.human_direction, "judge_direction": judge_dir,
                  "agrees": bool(judge_dir == c.human_direction),
                  "margin": round(abs(means[a] - means[b]), 3),
                  "verified": c.verified})

t8 = pd.DataFrame(rows8)
t8.to_csv("figdata/table8_judge_validation.csv", index=False)
print(f"judge reproduces {t8.agrees.sum()} of {len(t8)} published directional claims")
print(f"claims still awaiting a page cite: {(t8.verified != 'verified').sum()}")
t8

judge reproduces 2 of 3 published directional claims
claims still awaiting a page cite: 0


,claim_id,source_paper,comparison,human_direction,judge_direction,agrees,margin,verified
0,H1,Barile et al. (2024),most_pleasure vs average,b_fairer,b_fairer,True,0.100,verified
1,H2,Barile et al. (2024),most_pleasure vs least_misery,b_fairer,b_fairer,True,0.167,verified
2,H3,Barile et al. (2024),least_misery vs average,no_difference,a_fairer,False,0.067,verified


## Step 9 — what goes in the paper, and what must not

**Report, verbatim in the caption of every table above:**

> LLM-judge proxy (models = 3, k = 5, temperature = 0.0, prompt `v1`, drop
> rate = *X*%, order-bias discards = *Y*). No human judged any recommendation
> in this study.

**Report in the limitations:**

- the proxy is unvalidated against this system's own users;
- it is weaker than a study fine-tuned on real human survey responses;
- the mean pairwise inter-model Spearman, **including any negative pair** — if
  the families do not agree on the ordering, the pooled ranking is an average
  over disagreeing judges and the paper must say so;
- the unit of analysis is the item, not the call, so the CI is across groups
  (`n_items`), not across the `n_calls` column.

**Never write**, anywhere near these numbers: *participants rated*, *users
perceived*, *human evaluation*, *user study*.

### Raising this to the paper run

Set `N_GROUPS = 60` and leave `K = 5`. That is 60 x 4 x 3 x 5 x 2 = 7,200
calls. Nothing else changes — same prompt version, same seed, same selectors.
If the prompt is edited for any reason, bump `JUDGE_PROMPT_VERSION` in
`nbio.py` and re-collect everything; do not mix versions in one table.

## The Houston run with real diners (the paper's judge tables)

The cells above are a small demonstration on the five-restaurant catalog. The
paper's judge tables come from a larger run: Uber Eats Houston with Food.com
diners, prompt `v1`, seed 7, 24 groups of 3-4 diners, every selector, k = 5
repeats and both diner orders, rated by three judges from three families
(gpt-4o-mini through the OpenAI API, qwen3.8-27b and allam-2-7b through Groq).

Every call is appended to `runs/judge_v1_seed7_houston-tx-foodcom.jsonl`, and a
re-run skips every (model, group, selector, repeat, order) already in the log, so
it can be resumed. **The run is off by default**: with `JUDGE_LIVE = False` the
cell makes no API calls and the aggregation below reads the committed log. Setting
it to `True` needs `GROQ_API_KEY` and `OPENAI_API_KEY` and costs real money.
`JUDGE_LIMIT > 0` makes a smoke test of that many new calls per model.

In [15]:
# Parameters
JUDGE_LIVE = False     # True makes real, paid API calls and appends them to the log
JUDGE_LIMIT = 0        # >0: smoke test, that many new calls per model

In [16]:
import json, time, threading

HOU_SEED, HOU_N, HOU_K, HOU_TEMP = 7, 24, 5, 0.0
HOU_SELECTORS = ["average", "least_misery", "most_pleasure", "fair"]
HOU_LOG = os.path.join(nbio.HERE, "runs", f"judge_{nbio.JUDGE_PROMPT_VERSION}_seed{HOU_SEED}_houston-tx-foodcom.jsonl")


def run_houston_judge(limit=0):
    """Resumable judge run on Houston with Food.com diners. Makes API calls."""
    import openai
    from openai import OpenAI

    class Retrying:
        def __init__(self, c): self._c = c; self.chat = self; self.completions = self
        def create(self, **kw):
            for attempt in range(60):
                try: return self._c.chat.completions.create(**kw)
                except openai.RateLimitError as e:
                    wait = 15
                    try: wait = float(e.response.headers.get("retry-after", 15)) + 1
                    except Exception: pass
                    time.sleep(min(wait, 120))
                except (openai.APIConnectionError, openai.APITimeoutError, openai.InternalServerError):
                    time.sleep(10)
            raise RuntimeError("gave up after 60 retries")

    os.environ["FM_PERSONAS"] = "foodcom"
    os.environ["FM_DATASET"] = "ubereats"
    os.environ["FM_CITY"] = "Houston, TX"
    groq = Retrying(OpenAI(api_key=os.environ["GROQ_API_KEY"], base_url="https://api.groq.com/openai/v1", timeout=60))
    oai = Retrying(nbio.openai_client())   # capped by FM_OPENAI_BUDGET_USD
    models = {"gpt-4o-mini": oai, "qwen/qwen3.8-27b": groq, "allam-2-7b": groq}

    done_keys = set()
    if os.path.exists(HOU_LOG):
        for l in open(HOU_LOG):
            r = json.loads(l)
            done_keys.add((r["model"], r["group_id"], r["selector"], r["repeat"], r["reverse_order"]))

    cat = nbio.load_catalog()
    groups = nbio.generate_groups(HOU_N, seed=HOU_SEED, size_range=(3, 4))
    items = []
    for g in groups:
        for sel in HOU_SELECTORS:
            res = nbio.run_group(g, cat, method=sel)
            if res is None: continue
            opt = next(o for o in res["_F"] if o["option_id"] == res["pick_id"])
            items.append((g, sel, res, opt, res["_S"][res["pick_id"]]))
    lock = threading.Lock(); fh = open(HOU_LOG, "a"); done = {m: 0 for m in models}
    todo = {m: sum(1 for g, s, *_ in items for rep in range(HOU_K) for rev in (False, True)
                   if (m, g["group_id"], s, rep, rev) not in done_keys) for m in models}
    print(f"{len(items)} items; calls still to do: {todo}", flush=True)

    def worker(model):
        client = models[model]
        for g, sel, res, opt, sats in items:
            for rep in range(HOU_K):
                for rev in (False, True):
                    if (model, g["group_id"], sel, rep, rev) in done_keys: continue
                    if limit and done[model] >= limit: return
                    r = nbio.judge_recommendation(g, opt, sats, client=client, model=model, temperature=HOU_TEMP, reverse_order=rev)
                    r.update(group_id=g["group_id"], option_id=opt.get("option_id"), repeat=rep, selector=sel,
                             min_sat=res["min_sat"], jain=res["jain"], mean_sat=res["mean_sat"], option_name=opt["name"])
                    with lock:
                        fh.write(json.dumps(r) + "\n"); fh.flush(); done[model] += 1
                        if done[model] % 100 == 0: print(f"  {model}: {done[model]} new calls", flush=True)

    ts = [threading.Thread(target=worker, args=(m,)) for m in models]
    t0 = time.time(); [t.start() for t in ts]; [t.join() for t in ts]; fh.close()
    print(f"finished {sum(done.values())} new calls in {(time.time()-t0)/60:.1f} min -> {HOU_LOG}", flush=True)


if JUDGE_LIVE:
    run_houston_judge(JUDGE_LIMIT)
else:
    print(f"JUDGE_LIVE is False: no API calls. Using the committed log {os.path.relpath(HOU_LOG, nbio.HERE)}")

JUDGE_LIVE is False: no API calls. Using the committed log runs/judge_v1_seed7_houston-tx-foodcom.jsonl


## The Houston judge tables, from the log

The same aggregation as Steps 6-8, applied to the Houston log. The ensemble is
gpt-4o-mini, qwen3.8-27b and allam-2-7b; the log also holds partial gpt-oss-20b
rows (stopped by free-tier rate caps), which are excluded. To keep the design
balanced, only groups that every judge rated in full (40 calls each) are kept, so
every model sees every item.

Writes, into `figdata/ubereats-houston-tx-foodcom/`: `table7_judge.csv` (the
selector ranking, one mean per item with a 95% CI across items),
`table7_judge_per_model.csv`, `table7_judge_arith.csv` (the arithmetic metrics
of the same picks, for comparison) and `table8_judge_validation.csv` (agreement
with published human findings: "no_difference" claims agree when the two
fairness CIs overlap; directional claims need the judge's means to point the same
way and be separated).

In [17]:
import itertools
import pandas as pd

HOU_OUT = os.path.join(nbio.HERE, "figdata", "ubereats-houston-tx-foodcom")
HOU_MODELS = ["gpt-4o-mini", "qwen/qwen3.8-27b", "allam-2-7b"]
ORDER_TOLERANCE, UNSTABLE_SD = 1.0, 1.0
df = pd.DataFrame([json.loads(l) for l in open(HOU_LOG)])
df = df[df.model.isin(HOU_MODELS)].drop_duplicates(["model","group_id","selector","repeat","reverse_order"])
print("calls per model (all):", df.model.value_counts().to_dict())
# Balanced design: keep only groups every judge rated in full (40 calls each), so
# every model sees every item. Qwen stopped early on Groq's free-tier cap.
cnt = df[df.ok].groupby(["group_id","model"]).size().unstack(fill_value=0)
FULL = cnt.index[(cnt.reindex(columns=HOU_MODELS, fill_value=0) == 40).all(axis=1)]
df = df[df.group_id.isin(FULL)]
print(f"balanced groups: {len(FULL)}; calls per model:", df.model.value_counts().to_dict())
ok = df[df.ok]
print(f"drop rate {1-len(ok)/len(df):.3%} ({len(df)-len(ok)} of {len(df)})")
rank = ok.groupby(["model","selector"]).fairness.mean().unstack()[HOU_SELECTORS]
print(rank.round(3).to_string())
rhos = {}
for a, b in itertools.combinations(HOU_MODELS, 2):
    rhos[(a,b)] = nbio.spearman(rank.loc[a].tolist(), rank.loc[b].tolist()); print(f"rho {a} {b} {rhos[(a,b)]:+.3f}")
print(f"mean rho {sum(rhos.values())/3:+.3f}")
o = ok.groupby(["group_id","selector","reverse_order"])[["fairness","satisfaction","consensus"]].mean()
gaps = (o.xs(False, level="reverse_order") - o.xs(True, level="reverse_order")).abs()
flag = (gaps > ORDER_TOLERANCE).any(axis=1); DISC = set(gaps.index[flag])
print("order gaps:", gaps.mean().round(3).to_dict(), f"discarded {flag.sum()} of {len(gaps)}")
sd = ok.groupby(["group_id","selector","model"])[["fairness","satisfaction","consensus"]].std()
uns = (sd > UNSTABLE_SD).any(axis=1)
print("mean sd:", sd.mean().round(3).to_dict(), f"unstable {uns.sum()} of {len(sd)} ({uns.mean():.1%})")

calls per model (all): {'allam-2-7b': 960, 'gpt-4o-mini': 960, 'qwen/qwen3.8-27b': 587}


balanced groups: 14; calls per model: {'qwen/qwen3.8-27b': 560, 'allam-2-7b': 560, 'gpt-4o-mini': 560}
drop rate 0.000% (0 of 1680)
selector          average  least_misery  most_pleasure   fair
model                                                        
allam-2-7b          5.936         5.979          5.643  5.986
gpt-4o-mini         7.000         6.993          6.664  6.993
qwen/qwen3.8-27b    6.750         6.900          5.986  6.786
rho gpt-4o-mini qwen/qwen3.8-27b +0.316
rho gpt-4o-mini allam-2-7b +0.316
rho qwen/qwen3.8-27b allam-2-7b +0.800
mean rho +0.477
order gaps: {'fairness': 0.081, 'satisfaction': 0.123, 'consensus': 0.2} discarded 0 of 56


mean sd: {'fairness': 0.095, 'satisfaction': 0.086, 'consensus': 0.126} unstable 1 of 168 (0.6%)


In [18]:
keep = ok[~ok.set_index(["group_id","selector"]).index.isin(DISC)]
out = []
for sel in HOU_SELECTORS:
    s = keep[keep.selector == sel]; pi = s.groupby(["group_id","model"])[["fairness","satisfaction","consensus"]].mean().groupby("group_id").mean()
    row = {"selector": sel, "n_items": len(pi), "n_calls": len(s)}
    for d in ("fairness","satisfaction","consensus"):
        m, ci, _ = nbio.mean_ci(pi[d].tolist()); row[f"{d}_mean"], row[f"{d}_ci95"] = round(m,3), round(ci,3)
    row["drop_rate"] = round(1 - len(s)/max(1, len(df[df.selector == sel])), 4); out.append(row)
t7 = pd.DataFrame(out).sort_values("fairness_mean", ascending=False); t7.to_csv(f"{HOU_OUT}/table7_judge.csv", index=False)
print(t7.to_string())
pm = keep.groupby(["model","selector"])[["fairness","satisfaction","consensus"]].mean().round(3).reset_index()
pm.to_csv(f"{HOU_OUT}/table7_judge_per_model.csv", index=False)
# arithmetic metrics of the same picks, for comparison
ar = df.drop_duplicates(["group_id","selector"]).groupby("selector")[["min_sat","jain","mean_sat"]].mean().round(4)
print(ar.to_string()); ar.to_csv(f"{HOU_OUT}/table7_judge_arith.csv")

        selector  n_items  n_calls  fairness_mean  fairness_ci95  satisfaction_mean  satisfaction_ci95  consensus_mean  consensus_ci95  drop_rate
1   least_misery       14      420          6.624          0.057              6.464              0.133           6.564           0.094        0.0
3           fair       14      420          6.588          0.063              6.350              0.147           6.455           0.114        0.0
0        average       14      420          6.562          0.099              6.357              0.146           6.464           0.114        0.0
2  most_pleasure       14      420          6.098          0.360              5.998              0.297           5.960           0.355        0.0
               min_sat    jain  mean_sat
selector                                
average         0.8884  0.9966    0.9432
fair            0.8977  0.9979    0.9431
least_misery    0.9231  0.9995    0.9366
most_pleasure   0.7836  0.9788    0.8679


In [19]:
claims = pd.read_csv(os.path.join(nbio.HERE, "data", "judge", "human_claims.csv")); t7i = t7.set_index("selector")
# "no_difference" claims agree when the two fairness CIs overlap; directional claims
# need the judge's means to point the same way and be separated.
r8 = []
for _, c in claims.iterrows():
    a, b = t7i.loc[c.test_selector_a], t7i.loc[c.test_selector_b]
    gap = abs(a.fairness_mean - b.fairness_mean)
    sep = gap > a.fairness_ci95 + b.fairness_ci95
    jd = "no_difference" if not sep else ("a_fairer" if a.fairness_mean > b.fairness_mean else "b_fairer")
    r8.append({"claim_id": c.claim_id, "source_paper": c.source_paper,
               "comparison": f"{c.test_selector_a} vs {c.test_selector_b}",
               "human_direction": c.human_direction, "judge_direction": jd,
               "agrees": bool(jd == c.human_direction), "margin": round(gap, 3),
               "page_cite": c.page_cite, "verified": c.verified})
t8 = pd.DataFrame(r8); t8.to_csv(f"{HOU_OUT}/table8_judge_validation.csv", index=False); print(t8.to_string())

  claim_id          source_paper                     comparison human_direction judge_direction  agrees  margin                                         page_cite  verified
0       H1  Barile et al. (2024)       most_pleasure vs average        b_fairer        b_fairer    True   0.464  Study 1 Sect. 3.3 p. 15; Study 2 Sect. 4.3 p. 27  verified
1       H2  Barile et al. (2024)  most_pleasure vs least_misery        b_fairer        b_fairer    True   0.526                           Study 1 Sect. 3.3 p. 15  verified
2       H3  Barile et al. (2024)        least_misery vs average   no_difference   no_difference    True   0.062                           Study 1 Sect. 3.3 p. 15  verified
